# Explore the curated data with DuckDB

The pipeline exports `data/curated/parquet/*.parquet` after every run (Section 3.3.3). DuckDB queries them directly, so heavy analysis stays off PostgreSQL.

Run `pipeline demo` or `pipeline run` first.

In [ ]:
import duckdb
from pathlib import Path

PARQUET = Path('../data/curated/parquet')
con = duckdb.connect()
for name in ['weekly_cases', 'revision_history', 'alerts', 'forecasts']:
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{PARQUET / (name + '.parquet')}')")
con.sql('SELECT disease_name, count(*) AS weeks, sum(confirmed) AS confirmed FROM weekly_cases WHERE admin_level = 0 GROUP BY 1 ORDER BY 3 DESC')

In [ ]:
# Lassa fever: national weekly confirmed cases
lassa = con.sql("SELECT week_start, confirmed FROM weekly_cases WHERE disease_code = 'LASSA' AND admin_level = 0 ORDER BY week_start").df()
lassa.plot(x='week_start', y='confirmed', figsize=(12, 4), title='Lassa fever, weekly confirmed cases');

In [ ]:
# RQ2: how much and how late does NCDC revise figures?
con.sql('''SELECT field_name, count(*) AS revisions, round(avg(abs_change), 2) AS mean_abs_change,
                  round(median(revision_lag_weeks), 1) AS median_lag_weeks
           FROM revision_history GROUP BY 1 ORDER BY 2 DESC''')

In [ ]:
# Synthetic data check: never report results from demo data
con.sql('SELECT is_synthetic, count(*) FROM weekly_cases GROUP BY 1')